In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [2]:
df = pd.read_csv("Crop_recommendation.csv")  # change filename if needed

print(df.head())

    N   P   K  temperature   humidity        ph    rainfall label
0  90  42  43    20.879744  82.002744  6.502985  202.935536  rice
1  85  58  41    21.770462  80.319644  7.038096  226.655537  rice
2  60  55  44    23.004459  82.320763  7.840207  263.964248  rice
3  74  35  40    26.491096  80.158363  6.980401  242.864034  rice
4  78  42  42    20.130175  81.604873  7.628473  262.717340  rice


In [3]:
def soil_condition(row):
    if row['ph'] < 5 or row['ph'] > 8:
        return 2   # Polluted
    elif row['humidity'] < 40:
        return 1   # Moderate
    else:
        return 0   # Healthy

df['soil_label'] = df.apply(soil_condition, axis=1)

print(df[['ph', 'humidity', 'soil_label']].head())

         ph   humidity  soil_label
0  6.502985  82.002744           0
1  7.038096  80.319644           0
2  7.840207  82.320763           0
3  6.980401  80.158363           0
4  7.628473  81.604873           0


In [4]:
df = df.drop('label', axis=1)

In [5]:
X = df[['N', 'P', 'K', 'temperature', 'humidity', 'ph']]
y = df['soil_label']

In [6]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
model = Sequential()

model.add(Dense(8, input_dim=6, activation='relu'))
model.add(Dense(4, activation='relu'))
model.add(Dense(3, activation='softmax'))

C:\Users\shafa\AppData\Roaming\Python\Python310\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [10]:
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=10,
    validation_split=0.1
)

Epoch 1/50
159/159 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.5600 - loss: 0.9489 - val_accuracy: 0.8580 - val_loss: 0.7163
Epoch 2/50
159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8592 - loss: 0.5437 - val_accuracy: 0.8523 - val_loss: 0.4592
Epoch 3/50
159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8655 - loss: 0.3707 - val_accuracy: 0.8523 - val_loss: 0.3522
Epoch 4/50
159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8655 - loss: 0.2919 - val_accuracy: 0.8523 - val_loss: 0.2931
Epoch 5/50
159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9072 - loss: 0.2415 - val_accuracy: 0.9489 - val_loss: 0.2412
Epoch 6/50
159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9508 - loss: 0.1992 - val_accuracy: 0.9489 - val_loss: 0.1892
Epoch 7/50
159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9590 - loss: 0.1600 - val_accuracy: 0.9716 - val_loss: 0.1438
Epoch 8/50
159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9678 - loss: 0.1303 - val_accuracy: 0.

In [11]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9886 - loss: 0.0305
Test Accuracy: 0.9886363744735718


In [12]:
model.save("soil_model.h5")

In [13]:
import joblib
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [18]:

sample = [[1, 6, 3, 5, 2, 9]]  # extreme case

sample = scaler.transform(sample)
prediction = model.predict(sample)

print("Output:", np.argmax(prediction))

Soil Condition: Polluted
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step


C:\Users\shafa\AppData\Roaming\Python\Python310\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Output: 2
